In [2]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("spark://spark-master:7077")
         .appName("m1-dev")
         .config("spark.driver.memory", "1g")
         .config("spark.executor.memory", "1200m")
         .getOrCreate())

In [5]:
from functools import reduce
from pyspark.sql import DataFrame, functions as F

TYPES = {
        "tpep_pickup_datetime": "timestamp", 
        "tpep_dropoff_datetime": "timestamp",
        "PULocationID": "int", 
        "DOLocationID": "int", 
        "passenger_count": "int",
        "trip_distance": "double", 
        "RatecodeID": "int", 
        "payment_type": "int",
        "fare_amount": "double",
        "tip_amount": "double", 
        "total_amount": "double"
}

def months(spec):
    a, _, b = spec.partition("-")
    return range(int(a), int(b or a) + 1)

print(list(months("1-12")))
print(list(months("3")))


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
[3]


In [7]:
def load(spark, a):
    dfs = []
    for y in a.years:
        for m in months(a.months):
            df = spark.read.parquet(f"{a.input}/yellow_tripdata_{y}-{m:02d}.parquet")
            dfs.append(df.select(*[F.col(c).cast(t).alias(c) for c, t in TYPES.items()],
                                 F.lit(y).alias("year"), F.lit(m).alias("month")))
    return reduce(DataFrame.unionByName, dfs)

In [9]:
from types import SimpleNamespace
a = SimpleNamespace(input="/project/data/raw", years=[2017, 2019], months="1-1")

raw = load(spark, a)
raw.printSchema()
print("partições:", raw.rdd.getNumPartitions())
raw.groupBy("year").count().show()

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- year: integer (nullable = false)
 |-- month: integer (nullable = false)

partições: 12


+----+-------+
|year|  count|
+----+-------+
|2019|7696617|
|2017|9710820|
+----+-------+



In [10]:
def rules():
    c = F.col
    return {
        "null_critical": reduce(lambda x, y: x | y, [c(k).isNull() for k in TYPES
                                                     if k not in ("RatecodeID", "payment_type")]),
        "fare_le_0": c("fare_amount") <= 0,
        "dist_le_0": c("trip_distance") <= 0,
        "pax_0_or_gt6": (c("passenger_count") == 0) | (c("passenger_count") > 6),
        "dropoff_le_pickup": c("tpep_dropoff_datetime") <= c("tpep_pickup_datetime"),
        "loc_out_of_range": ~c("PULocationID").between(1, 263) | ~c("DOLocationID").between(1, 263),
    }

In [11]:
r = {k: F.coalesce(v, F.lit(False)) for k, v in rules().items()}

raw.groupBy("year").agg(
    F.count("*").alias("total"),
    *[F.sum(v.cast("int")).alias(k) for k, v in r.items()]
).show(vertical=True)

-RECORD 0--------------------
 year              | 2017    
 total             | 9710820 
 null_critical     | 0       
 fare_le_0         | 6696    
 dist_le_0         | 58678   
 pax_0_or_gt6      | 605     
 dropoff_le_pickup | 9343    
 loc_out_of_range  | 197915  
-RECORD 1--------------------
 year              | 2019    
 total             | 7696617 
 null_critical     | 28672   
 fare_le_0         | 9770    
 dist_le_0         | 55089   
 pax_0_or_gt6      | 117438  
 dropoff_le_pickup | 6557    
 loc_out_of_range  | 187558  



In [12]:
def run(spark, a, opt, prof):
    zones = spark.read.csv(a.zones, header=True,
                           schema="LocationID INT, Borough STRING, Zone STRING, service_zone STRING")
    with prof.profile("M1"):
        raw = load(spark, a)
        if a.fraction < 1:
            raw = raw.sample(fraction=a.fraction, seed=42)
        r = {k: F.coalesce(v, F.lit(False)) for k, v in rules().items()}
        stats = raw.groupBy("year").agg(F.count("*").alias("total"),
                    *[F.sum(v.cast("int")).alias(k) for k, v in r.items()],
                    F.sum(reduce(lambda x, y: x | y, r.values()).cast("int")).alias("removed_any"))
        stats.coalesce(1).write.mode("overwrite").json(f"{a.output}/m1_stats")
        trips = raw.filter(~reduce(lambda x, y: x | y, r.values()))
    return trips, zones

In [13]:
r = {k: F.coalesce(v, F.lit(False)) for k, v in rules().items()}
any_bad = reduce(lambda x, y: x | y, r.values())

stats = raw.groupBy("year").agg(F.count("*").alias("total"),
            *[F.sum(v.cast("int")).alias(k) for k, v in r.items()],
            F.sum(any_bad.cast("int")).alias("removed_any"))
stats.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[year#30], functions=[count(1), sum(cast(((((((((isnull(tpep_pickup_datetime#19) OR isnull(tpep_dropoff_datetime#20)) OR isnull(PULocationID#21)) OR isnull(DOLocationID#22)) OR isnull(passenger_count#23)) OR isnull(trip_distance#24)) OR isnull(fare_amount#27)) OR isnull(tip_amount#28)) OR isnull(total_amount#29)) as int)), sum(cast(coalesce((fare_amount#27 <= 0.0), false) as int)), sum(cast(coalesce((trip_distance#24 <= 0.0), false) as int)), sum(cast(coalesce(((passenger_count#23 = 0) OR (passenger_count#23 > 6)), false) as int)), sum(cast(coalesce((tpep_dropoff_datetime#20 <= tpep_pickup_datetime#19), false) as int)), sum(cast(coalesce((((PULocationID#21 < 1) OR (PULocationID#21 > 263)) OR ((DOLocationID#22 < 1) OR (DOLocationID#22 > 263))), false) as int)), sum(cast((((((((((((((isnull(tpep_pickup_datetime#19) OR isnull(tpep_dropoff_datetime#20)) OR isnull(PULocationID#21)) OR isnull(DOLocationID#22)) OR i

In [14]:
stats.coalesce(1).write.mode("overwrite").json("/project/output/m1_stats")

trips = raw.filter(~any_bad)
print("limpas:", trips.count())
spark.read.json("/project/output/m1_stats").show(vertical=True)

limpas: 16796333


-RECORD 0--------------------
 dist_le_0         | 55089   
 dropoff_le_pickup | 6557    
 fare_le_0         | 9770    
 loc_out_of_range  | 187558  
 null_critical     | 28672   
 pax_0_or_gt6      | 117438  
 removed_any       | 369508  
 total             | 7696617 
 year              | 2019    
-RECORD 1--------------------
 dist_le_0         | 58678   
 dropoff_le_pickup | 9343    
 fare_le_0         | 6696    
 loc_out_of_range  | 197915  
 null_critical     | 0       
 pax_0_or_gt6      | 605     
 removed_any       | 241596  
 total             | 9710820 
 year              | 2017    

